# 🔧 Model Embeddings Configuration Manager for RAG Analysis

This notebook allows you to create, manage, and select between different **LLM embedding model configurations** for use with tools that create embeddings.

It mirrors the existing LLM configuration mechanism but is **strictly scoped**
to embedding models (vectorization, retrieval, file indexing).

Each configuration includes:
- 🤖 **Model name** (e.g., `text-embedding-3-large`, `text-embedding-3-small`)
- 🌐 **Base URL** (e.g., `https://api.openai.com/v1``)
- 🔐 **API Key** (e.g., `sk-...` or `ollama`, `sk-noauth`)

---

## 🗃️ Configuration Storage

All configurations are stored in the following JSON file:

```
~/.secrets/model_configs.json
```

This file will look something like:

```json
{
  "_default": "openai_large",
  "openai_large": {
    "provider": "openai",
    "api_key": "${OPENAI_API_KEY}",
    "base_url": "${OPENAI_BASE_URL}",
    "embedding_model": "text-embedding-3-large",
    "dimension": 3072
  },
  "openai_small": {
    "provider": "openai",
    "api_key": "${OPENAI_API_KEY}",
    "base_url": "${OPENAI_BASE_URL}",
    "embedding_model": "text-embedding-3-small",
    "dimension": 1536
  }
}

```

You can **manually edit or copy** this file if needed to:
- Share configurations with others
- Maintain consistent setups across environments
- Set or change the default model (`"_default"` key)

---

## ✅ How to Use This Notebook

1. Run the notebook cells to:
   - Add or update a named model configuration
   - Optionally set it as the default

2. Use your configurations in tools like `ChatGPTAnalyzer`:

```python
from Open_AI_RAG_manager import ChatGPTAnalyzer

# Use specific model
analyzer = ChatGPTAnalyzer.from_config("mistral", yaml_content=my_yaml)

# Or just use the default model
analyzer = ChatGPTAnalyzer.from_config(yaml_content=my_yaml)
```

---

> 💡 Tip: You can define as many configurations as you like — great for testing different LLMs or backends like OpenAI, LM Studio, Ollama, and Hugging Face APIs.



In [ ]:
import json
from pathlib import Path
from getpass import getpass
from IPython.display import display, Markdown

CONFIG_FILE = Path.home() / ".secrets" / "embeddings_configs.json"
CONFIG_FILE.parent.mkdir(exist_ok=True)

DEFAULT_KEY = "_default"

def load_configs():
    if CONFIG_FILE.exists():
        with CONFIG_FILE.open() as f:
            return json.load(f)
    return {}

def save_configs(configs):
    with CONFIG_FILE.open("w") as f:
        json.dump(configs, f, indent=2)

def get_config_names(configs):
    return sorted([k for k in configs.keys() if k != DEFAULT_KEY])

def list_configs():
    configs = load_configs()
    names = get_config_names(configs)

    if not names:
        display(Markdown("❌ **No configurations found.**"))
        return

    default_name = configs.get(DEFAULT_KEY, "None")
    display(Markdown(f"### 📦 Saved Configurations (default: `{default_name}`)"))
    for name in names:
        cfg = configs[name]
        display(Markdown(f"- `{name}` → model: `{cfg.get('model','')}`, url: `{cfg.get('base_url','')}`"))

def set_default_config():
    configs = load_configs()
    names = get_config_names(configs)
    if not names:
        display(Markdown("❌ **No configurations to set as default.**"))
        return

    list_configs()
    name = input("⭐ Enter config name to set as default: ").strip()
    if name not in configs or name == DEFAULT_KEY:
        display(Markdown(f"❌ **Config `{name}` not found.**"))
        return

    configs[DEFAULT_KEY] = name
    save_configs(configs)
    display(Markdown(f"🌟 **Default configuration set to `{name}`.**"))

def create_or_update_config():
    name = input("🔖 Enter configuration name (e.g., mistral): ").strip()
    if not name or name == DEFAULT_KEY:
        display(Markdown(f"❌ **Invalid name. `{DEFAULT_KEY}` is reserved.**"))
        return

    configs = load_configs()
    existing = configs.get(name, {})

    model = input(f"🤖 Model name [{existing.get('model','')}]: ").strip() or existing.get("model", "")
    base_url = input(f"🌐 Base URL [{existing.get('base_url','')}]: ").strip() or existing.get("base_url", "")

    # Don’t echo existing API key. Let user keep or replace.
    replace_key = input("🔐 Replace API Key? (y/N): ").strip().lower() == "y"
    if replace_key or "api_key" not in existing:
        api_key = getpass("🔐 API Key (or sk-noauth): ").strip()
    else:
        api_key = existing.get("api_key", "")

    configs[name] = {"model": model, "base_url": base_url, "api_key": api_key}

    # Optional: set default right here
    make_default = input("⭐ Set this config as default? (y/N): ").strip().lower() == "y"
    if make_default:
        configs[DEFAULT_KEY] = name

    save_configs(configs)
    display(Markdown(f"✅ **Configuration `{name}` saved.**"))
    if make_default:
        display(Markdown("🌟 **Set as default configuration.**"))

def delete_config():
    configs = load_configs()
    names = get_config_names(configs)
    if not names:
        display(Markdown("❌ **No configurations to delete.**"))
        return

    list_configs()
    name = input("🗑️ Enter config name to delete: ").strip()
    if name not in configs or name == DEFAULT_KEY:
        display(Markdown(f"❌ **Config `{name}` not found.**"))
        return

    confirm = input(f"⚠️ Really delete `{name}`? (y/N): ").strip().lower() == "y"
    if not confirm:
        display(Markdown("✅ **Delete cancelled.**"))
        return

    # Remove it
    configs.pop(name, None)

    # If it was default, clear or pick a new default
    if configs.get(DEFAULT_KEY) == name:
        remaining = get_config_names(configs)
        if remaining:
            configs[DEFAULT_KEY] = remaining[0]
            display(Markdown(f"ℹ️ **Default deleted; new default is `{remaining[0]}`.**"))
        else:
            configs.pop(DEFAULT_KEY, None)
            display(Markdown("ℹ️ **Default deleted; no configs remain.**"))

    save_configs(configs)
    display(Markdown(f"🗑️ **Deleted `{name}`.**"))

def rename_config():
    configs = load_configs()
    names = get_config_names(configs)
    if not names:
        display(Markdown("❌ **No configurations to rename.**"))
        return

    list_configs()
    old = input("✏️ Enter config name to rename: ").strip()
    if old not in configs or old == DEFAULT_KEY:
        display(Markdown(f"❌ **Config `{old}` not found.**"))
        return

    new = input("✏️ Enter new name: ").strip()
    if not new or new == DEFAULT_KEY or new in configs:
        display(Markdown("❌ **Invalid new name (reserved, empty, or already exists).**"))
        return

    configs[new] = configs.pop(old)
    if configs.get(DEFAULT_KEY) == old:
        configs[DEFAULT_KEY] = new

    save_configs(configs)
    display(Markdown(f"✅ **Renamed `{old}` → `{new}`.**"))

def menu():
    while True:
        display(Markdown(
            "## 🔧 Embeddings Config Manager\n"
            "Choose an action:\n"
            "1) List configs\n"
            "2) Create/Update config\n"
            "3) Set default config\n"
            "4) Delete config\n"
            "5) Rename config\n"
            "0) Exit"
        ))
        choice = input("➡️ Enter choice: ").strip()

        if choice == "1":
            list_configs()
        elif choice == "2":
            create_or_update_config()
        elif choice == "3":
            set_default_config()
        elif choice == "4":
            delete_config()
        elif choice == "5":
            rename_config()
        elif choice == "0":
            display(Markdown("👋 Done."))
            break
        else:
            display(Markdown("❌ **Invalid choice.**"))

# Run the manager
menu()


## 🔧 Embeddings Config Manager
Choose an action:
1) List configs
2) Create/Update config
3) Set default config
4) Delete config
5) Rename config
0) Exit

➡️ Enter choice:  1


### 📦 Saved Configurations (default: `open_ai_large`)

- `Qwen_emb` → model: `Qwen3-Embedding-8B`, url: `https://api.siemens.com/llm/v1`

- `open_ai_large` → model: `text-embedding-3-large`, url: `https://api.openai.com/v1`

...

🌟 **Default configuration set to `open_ai_large`.**

## 🔧 Embeddings Config Manager
Choose an action:
1) List configs
2) Create/Update config
3) Set default config
4) Delete config
5) Rename config
0) Exit

➡️ Enter choice:  6


❌ **Invalid choice.**

## 🔧 Embeddings Config Manager
Choose an action:
1) List configs
2) Create/Update config
3) Set default config
4) Delete config
5) Rename config
0) Exit

In [ ]:
 Path.home()

In [ ]:
print( Path.home())

In [ ]:
cd .secrets